In [ ]:
import pandas as pd
import numpy as np
import os
import jieba
# import time
# import matplotlib.pyplot as plt
import seaborn as sns
# import statsmodels.sapi as sm
# import tqdm
import datetime
import gc

# import jionlp as jio
# from textrank4zh import TextRank4Keyword
# from hanlp_restful import HanLPClient
from gensim.models.coherencemodel import CoherenceModel
from snownlp import sentiment
from snownlp import SnowNLP
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from collections import Counter
# from gensim.similarities import MatrixSimilarity

sns.set_style('white')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 一、数据读取

In [ ]:
import os
import pandas as pd
import datetime
import numpy as np

In [ ]:
# 数据读取
def data_import(path = '../dataset/互动易new/'):
    names = os.listdir(path)
    if '.DS_Store' in names:
        names.remove('.DS_Store')
    print(f'We have {len(names)} stocks Q&A data')
    all_qa_comments = sum([int(name.split('_')[2].split('.')[0]) for name in names])
    print(f'The total Q&A comments are {all_qa_comments}')
    df = [pd.read_csv(path + name, dtype={'stock_code':str}) for name in names]
    df = pd.concat(df, axis=0)

    # 删除ques_text和ans_text都是nan的值
    df.dropna(axis=0, subset=['ques_text', 'ans_text'], how='all', inplace=True)

    # 将回答为空值标记为字符串0
    df.fillna('n', inplace=True)

    return df


## 1.1 文本指标处理
### 1.1.1 指标读取


In [ ]:
# path = '../dataset/互动易/'
# names = os.listdir(path)
# print(f'We have {len(names)} stocks Q&A data')
# all_qa_comments = sum([int(name.split('_')[2].split('.')[0]) for name in names])
# print(f'The total Q&A comments are {all_qa_comments}')

# df = [pd.read_csv(path + name, dtype={'stock_code':str}) for name in names]
# df = pd.concat(df, axis=0)

# # 删除ques_text和ans_text都是nan的值
# df.dropna(axis=0, subset=['ques_text', 'ans_text'], how='all', inplace=True)

# # 将回答为空值标记为字符串0
# df.fillna('n', inplace=True)

In [ ]:
# 读取数据
df = data_import()

# 字数指标构建
df ['ques_lenth'] = df['ques_text'].apply(lambda x: len(x) if type(x) == str else 0)
df ['ans_length'] = df['ans_text'].apply(lambda x: len(x) if type(x) == str else 0)

### 1.1.2 日回答数目的对数构建

In [ ]:
# 计算当天问题数和回答数
pubDate = df[['stock_code', 'pubDate']]
ansDate = df.loc[df['ansDate']!='n', ['stock_code', 'ansDate']]

pubDate['pubDate'] = (pubDate['pubDate']/1000).apply(lambda x: datetime.datetime.fromtimestamp(x)).dt.date
ansDate['ansDate'] = (ansDate['ansDate']/1000).apply(lambda x: datetime.datetime.fromtimestamp(x)).dt.date

# 设置日期为index
pubDate.set_index('pubDate', inplace=True)
ansDate.set_index('ansDate', inplace=True)

# dt.date之后的格式为object，需要重新转换为datetime64
pubDate.index = pd.to_datetime(pubDate.index)
ansDate.index = pd.to_datetime(ansDate.index)


pubDate = pubDate.groupby('stock_code').resample('D').size().reset_index()
ansDate = ansDate.groupby('stock_code').resample('D').size().reset_index()

pubDate.rename(columns={0:'ques_num', 'pubDate':'date'}, inplace=True)
ansDate.rename(columns={0:'ans_num', 'ansDate':'date'}, inplace=True)

ddf = pd.merge(pubDate, ansDate, how='outer', left_on=['stock_code', 'date'], right_on=['stock_code', 'date'])
ddf.fillna(0, inplace=True)

# 构造变量
ddf['ques_num'] = np.log(ddf['ques_num']+1)
ddf['ans_num'] = np.log(ddf['ans_num']+1)

# 构造新的stock_code 格式
ddf['stock_code'] = ddf['stock_code'].apply(lambda x: 'SZ' + x)

ddf.rename(columns={'stock_code':'instrument', 'date':'datetime'}, inplace=True)

ddf.head()
ddf.to_csv('../dataset/中间过程数据/1127_log_textNum.csv', index=False)

In [ ]:
# ddf = df[['pubDate', 'stock_code', 'ansDate']].copy()
# ddf['ques_num'] = df['ques_text'].apply(lambda x: 1 if len(x) > 1 else 0)
# ddf['ans_num'] = df['ans_text'].apply(lambda x: 1 if len(x) > 1 else 0)

# # 将unix时间转化为日度时


# ddf['pubDate'] = (ddf['pubDate']/1000).apply(lambda x: datetime.datetime.fromtimestamp(x))
# ddf['ansDate'] = (ddf['ansDate']/1000).apply(lambda x: datetime.datetime.fromtimestamp(x))

# ddf.set_index('pubDate', inplace=True)

# # 生成完整的日期序列
# dates = pd.date_range(ddf['pubDate'].min(). ddf['pubDate'].max(), freq='D')
# ddf = ddf.reindex(dates, fill_value=0)

# # # 按日度聚合
# # ddf = ddf.groupby('stock_code').resample('D').sum()

# # # 删除outer的行
# # ddf = ddf[ddf['stock_code']!=0]
# # ddf = ddf.drop(columns=['stock_code']).reset_index()

# # # # 转化为ln形式
# # ddf['ques_num'] = np.log(ddf['ques_num']+1)
# # ddf['ans_num'] = np.log(ddf['ans_num']+1)

# # # 输出
# # ddf.to_csv('../dataset/中间过程数据/0428_log_textNum.csv')

### 1.1.3 情绪因子构建

In [ ]:
filepath_pos_vocab = '../dataset/预训练语料库/pos_vocab.txt'
filepath_neg_vocab = '../dataset/预训练语料库/neg_vocab.txt'

sentiment.train(filepath_neg_vocab, filepath_pos_vocab)

In [ ]:
def sentiment_score(text):
    s = SnowNLP(text)
    return s.sentiments

# 问题与回答情绪情绪指标构建
# 耗时1min9s
df['ques_score'] = df['ques_text'].apply(lambda x: sentiment_score(x))
df['ans_score'] = df['ans_text'].apply(lambda x: sentiment_score(x))

### 1.1.4 回答相似度

使用tf-idf提取纯粹考虑用向量化表示后二者之间主题的联系，但是并未考虑语义的相关性，有待进一步优化

In [ ]:
# 初始化TF-IDF向量器
tfidf_vectorizer = TfidfVectorizer()
# 合并问题和答案文本，以便一起转换
merged_texts = df['ques_text'].tolist() + df['ans_text'].tolist()
# 计算TF-IDF
tfidf_matrix = tfidf_vectorizer.fit_transform(merged_texts)

# 分割回问题和答案的TF-IDF向量
ques_tfidf = tfidf_matrix[:len(df)]
ans_tfidf = tfidf_matrix[len(df):]

# 计算问题与对应答案之间的余弦相似度
similarities = cosine_similarity(ques_tfidf, ans_tfidf).diagonal()

# 添加相似度到原DataFrame中
df['Similarity'] = similarities

# 耗时2min18s

### 1.1.5日期转化与日度聚合

In [ ]:
# 日期转化
df['pubDate'] = (df['pubDate']/1000).apply(lambda x: datetime.datetime.fromtimestamp(x))
df.to_csv('../dataset/中间过程数据/timestamp_sentiment_similarity.csv')

In [ ]:
# 日度聚合——平均值
# df.set_index('pubDate', inplace=True)

# 聚合
ndf = df[['stock_code', 'ques_lenth', 'ans_length', 'ques_score', 'ans_score', 'Similarity']].groupby('stock_code').resample('D').mean()
ndf = ndf.drop(columns=['stock_code']).dropna().reset_index()
ndf.to_csv('../dataset/中间过程数据/daily_sentiment_similarity.csv')

# 二、 文本LDA建模
# 2.1 数据读取

In [ ]:
# 读取数据 
df = data_import()

# 读取停用词列表
with open('../dataset/预训练语料库/stopwords.txt', 'r') as file:
    stopwords = file.readlines()
    stopwords = [x.strip() for x in stopwords]

# 添加空白字符在停用词中
stopwords = set([' ', '\n', '\r', '\r\n'] + list(stopwords))

## 2.2 分词与去除停用词
### 2.2.1 jieba分词

In [ ]:
# 问题分词(1min14s)
ques_text = ''.join(df['ques_text'].tolist())
ques_words = [word for word in jieba.cut(ques_text) if word not in stopwords]
ans_text = ''.join(df['ans_text'].tolist())
ans_words = [word for word in jieba.cut(ans_text) if word not in stopwords]


In [ ]:
# 分词效果展示
# 问题词频top50
print('------------------问题词频top50------------------')
elements_counts = Counter(ques_words)
most_elements = elements_counts.most_common(20)
for element, count in most_elements:
    print(f'{element}:{count}')

print('------------------回答词频top50------------------')
# 回答词频top50
elements_counts = Counter(ans_words)
most_elements = elements_counts.most_common(20)
for element, count in most_elements:
    print(f'{element}:{count}')

### 2.2.2 jionlp

In [ ]:
# # 运行市场超过10min，舍弃
# ques_words = jio.keyphrase.extract_keyphrase(ques_text,50)
# ans_keywords = jio.keyphrase.extract_keyphrase(ans_text,50)

## 2.3 LDA模型

In [ ]:
# question主题搜索6min26s
# 建立词典
dictionary = Dictionary([ques_words])

# 创建语料库
corpus = [dictionary.doc2bow(ques_words)]

# 采用一致性评分计算最优主题数
coherence_score = []
# lda模型
for i in range(2, 11):
    lda = LdaModel(corpus, num_topics=i, id2word=dictionary, random_state=2024, chunksize=10, passes=15, alpha='auto', eta='auto')
    coherencemodel = CoherenceModel(model=lda, texts=[ques_words], dictionary=dictionary, coherence='c_v')
    coherence_score.append(coherencemodel.get_coherence())
coherence_score

In [ ]:
lda = LdaModel(corpus, num_topics=coherence_score.index(min(coherence_score))+2, id2word=dictionary, random_state=2024, chunksize=10, passes=15, alpha='auto', eta='auto')
topics = lda.print_topics(num_words=coherence_score.index(min(coherence_score))+2)
topics

In [ ]:
# 上述搜索结果并不好
lda = LdaModel(corpus, num_topics=3, id2word=dictionary, random_state=2024, chunksize=10, passes=15, alpha='auto', eta='auto')
topics = lda.print_topics(num_words=5)
topics

In [ ]:
# 答案ans主题
# 建立词典
dictionary = Dictionary([ans_words])

# 创建语料库
corpus = [dictionary.doc2bow(ques_words)]

# lda模型
lda = LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15)

topics = lda.print_topics(num_words=5)
topics

In [ ]:
# 总结，主题词模型提取结果很糟糕
# 词频尚可使用
# 或许可以按年份将LDA权重较大的词汇提取出来统计随年份变化的词频

## TEST

In [ ]:
# 导入停用词表
with open('../dataset/语料库/stopwords.txt', 'r') as file:
    stopwords = file.readlines()
    stopwords = [x.strip() for x in stopwords]

# text = df['ques_text'].tolist() + df['ans_text'].tolist()
# s = ''.join(text)
# text = [word for word in jieba.lcut(s) if word not in stopwords]
# # 问题与回答分词并剔除停用词
# ques_text = df['ques_text'].apply(lambda x: [word for word in jieba.cut(x) if word not in stopwords])
# ans_text = df['ans_text'].apply(lambda x: [word for word in jieba.cut(x) if word not in stopwords])
# # 创建字典和语料库
# texts = ques_text.tolist() + ans_text.tolist()

# # 创建字典
# dictionary = Dictionary(texts)

# # 创建BoW表示
# corpus = [dictionary.doc2bow(text) for text in texts]

# # 使用TF-IDF模型
# tfidf = TfidfModel(corpus)
# corpus_tfidf = tfidf[corpus]

# # 为了计算相似度，我们需要将问题和答案分开
# ques_vecs = corpus_tfidf[:df.shape[0]]
# ans_vecs = corpus_tfidf[df.shape[0]:]
# # 使用MatrixSimilarity计算相似度
# sim = []
# index = MatrixSimilarity(ans_vecs)  # 以答案作为参考

# for i, ques_vec in tqdm.tqdm(enumerate(ques_vecs)):
#     sims = index[ques_vec]  # 计算当前问题与所有答案的相似度
#     sim.append(sims[i])

In [ ]:
text = df['ques_text'].tolist() + df['ans_text'].tolist()
s = ''.join(text)
text = [word for word in jieba.lcut(s if word not in stopwords)]


In [ ]:
t0 = pd.to_datetime('2010-01-01') + pd.Timedelta(days=365)

In [ ]:
path1 = '../dataset/互动易new'
path2 = '../dataset/互动易old'
ostk = os.listdir(path2)
nstk = os.listdir(path1)
nstk.remove('.DS_Store')
ostk = set([i.split('.')[0].split('_')[1] for i in ostk])
nstk = set([i.split('.')[0].split('_')[1] for i in nstk])

In [ ]:
ddf[(ddf['ques_num']==0)&(ddf['ans_num']>0)].sort_values('date').head(50)